In [ ]:
%%capture
!pip install darts neuralforecast gluonts stengression pytorchts

# Standard library imports
import os
import time
import random
import subprocess

# Data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler

# PyTorch and deep learning imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import TensorDataset, DataLoader, Dataset
from einops import rearrange

# Darts imports
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import (
    NHiTSModel, 
    RNNModel, 
    BlockRNNModel, 
    ConformalNaiveModel, 
    ConformalQRModel,
    TimesFM2p5Model, 
    Chronos2Model, 
    PatchTSTFMModel
)
from darts.utils.likelihood_models import QuantileRegression, GaussianLikelihood

# NeuralForecast imports
from neuralforecast import NeuralForecast
from neuralforecast.models import DeepNPTS
from neuralforecast.utils import PredictionIntervals

# GluonTS imports
from gluonts.dataset.common import ListDataset
from gluonts.dataset.field_names import FieldName
from gluonts.evaluation.backtest import make_evaluation_predictions

# Stengression imports
from stengression import (
    GraphInfo, 
    energy_score_loss, 
    GCEN, 
    SpatioTemporalDataset, 
    compute_adjacency_matrix
)

# Metrics

In [ ]:
# Metrics
"""
Args:
y_true (torch.Tensor): Ground truth for the forecast horizon of shape 
    :math:`(T_{out}, N, D_{in})`.
y_train (torch.Tensor): In-sample training data of shape 
    :math:`(T_{train}, N, D_{in})`. Required for scaling MASE and RMSSE.
"""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def mae(y_true, y_pred):
    return torch.mean(torch.abs(y_true - y_pred)).item()

def pinball_loss(y_true, y_pred_quantile, quantile):
    error = y_true - y_pred_quantile
    return torch.mean(torch.max(quantile * error, (quantile - 1) * error))

def crps_approximation(y_true, y_preds_ensemble):
    # CRPS = E|X - y| - 0.5 E|X - X'|
    abs_diff_true = torch.mean(torch.abs(y_preds_ensemble - y_true), dim=0)
    abs_diff_samples = torch.mean(torch.abs(y_preds_ensemble.unsqueeze(0) - y_preds_ensemble.unsqueeze(1)), dim=(0,1))
    crps = torch.mean(abs_diff_true - 0.5 * abs_diff_samples)
    return crps.item()

def mase(y_true, y_pred, y_train):
    # Calculate MAE for forecast period across all elements
    mae_forecast = torch.mean(torch.abs(y_true - y_pred))

    # Calculate scaling factor: mean absolute difference of y_train across all elements
    diff = torch.abs(y_train[1:] - y_train[:-1])
    scale = torch.mean(diff)
    scale = scale if scale != 0 else 1e-8

    mase_score = mae_forecast / scale

    return mase_score.item()

def rmsse(y_true, y_pred, y_train):
    # Calculate MSE for forecast period across all elements
    mse_forecast = torch.mean((y_true - y_pred)**2)

    # Calculate scaling factor: mean squared difference of y_train across all elements
    diff = y_train[1:] - y_train[:-1]
    scale = torch.mean(diff ** 2)
    scale = scale if scale != 0 else 1e-8

    rmsse_score = torch.sqrt(mse_forecast / scale)

    return rmsse_score.item()


def empirical_coverage(true_values, lower_bounds, upper_bounds):
    """
    Calculate empirical coverage probability (PICP) that true values lie within predicted CIs.

    Args:
        true_values (array-like): True target values.
        lower_bounds (array-like): Lower bounds of predicted confidence intervals.
        upper_bounds (array-like): Upper bounds of predicted confidence intervals.

    Returns:
        float: Fraction of true values inside the confidence intervals.
    """
    true_values = np.array(true_values)
    lower_bounds = np.array(lower_bounds)
    upper_bounds = np.array(upper_bounds)

    inside = (true_values >= lower_bounds) & (true_values <= upper_bounds)
    coverage = np.mean(inside)
    return coverage


def winkler_score(true_values, lower_bounds, upper_bounds, alpha):
    """
    Calculate average Winkler score for prediction intervals.

    Parameters:
    true_values (array-like): Ground truth values.
    lower_bounds (array-like): CI lower bounds
    upper_bounds (array-like): CI upper bounds.
    alpha (float): Significance level (e.g. 0.05 for 95% CI).

    Returns:
    float: Average Winkler score.
    """
    true_values = np.array(true_values)
    lower_bounds = np.array(lower_bounds)
    upper_bounds = np.array(upper_bounds)

    widths = upper_bounds - lower_bounds
    scores = widths.copy()

    below = true_values < lower_bounds
    above = true_values > upper_bounds

    scores[below] += (2 / alpha) * (lower_bounds[below] - true_values[below])
    scores[above] += (2 / alpha) * (true_values[above] - upper_bounds[above])

    return np.mean(scores)

In [ ]:
# Set your data path here, which should be of shape (T, N) or (time_steps, num_nodes)
# This file was created on the Belgium COVID-19 dataset, so the context and prediction lengths are set accordingly
# They need to be changed as per your requirements
data_path = 'set_your_path_here.csv'

# 1. Pre-Control Limits: LSTM-PC

In [ ]:
df = pd.read_csv(data_path)
num_nodes = df.shape[1]
series = TimeSeries.from_dataframe(df)

horizon = 30
train, val = series[:-horizon], series[-horizon:]

# Standardize
scaler = Scaler()
train_scaled = scaler.fit_transform(train)

# Model training
print(f"Training LSTM model on shape {df.shape} with standardized data...")
model = BlockRNNModel(
    model="LSTM", 
    hidden_dim=64,
    n_rnn_layers=2,
    input_chunk_length=60,
    output_chunk_length=horizon,
    n_epochs=100,
    random_state=42,
    pl_trainer_kwargs={"enable_progress_bar": True, "logger": False}
)

start_train = time.time()
model.fit(train_scaled)
training_time = time.time() - start_train

print("Generating in-sample predictions for residuals...")
in_sample_preds_scaled = model.historical_forecasts(
    train_scaled,
    start=model.input_chunk_length,
    forecast_horizon=1,
    retrain=False,
    verbose=False
)

# Invert transform to calculate residuals on the original scale
in_sample_preds = scaler.inverse_transform(in_sample_preds_scaled)
in_sample_true = train.slice_intersect(in_sample_preds)

residuals = in_sample_true.values() - in_sample_preds.values()
sigma_hat = np.std(residuals, axis=0) 

print("Generating out-of-sample forecast...")
start_infer = time.time()
point_forecast_scaled = model.predict(n=horizon)
inference_time = time.time() - start_infer

# Invert transform for final predictions
point_forecast = scaler.inverse_transform(point_forecast_scaled)

y_pred_np = point_forecast.values()
y_true_np = val.values()
y_train_np = train.values() # Keep original scale for baseline metric calculations
    
# Pre-control Limit Methodology
alpha_winkler = 0.05
z_975 = norm.ppf(1 - alpha_winkler/2)
lower_bounds_95_np = y_pred_np - (z_975 * sigma_hat)
upper_bounds_95_np = y_pred_np + (z_975 * sigma_hat)

q_80 = 0.8
z_80 = norm.ppf(q_80)
pred_80_np = y_pred_np + (z_80 * sigma_hat)

k = 1.5
lower_bounds_k15_np = y_pred_np - (k * sigma_hat)
upper_bounds_k15_np = y_pred_np + (k * sigma_hat)

# Metrics
def format_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32).unsqueeze(-1)

y_true_t = format_tensor(y_true_np)
y_pred_t = format_tensor(y_pred_np)
y_train_t = format_tensor(y_train_np)
pred_80_t = format_tensor(pred_80_np)

n_samples = 100
sigma_tensor = torch.tensor(sigma_hat, dtype=torch.float32).view(1, 1, num_nodes, 1)
noise = torch.randn(n_samples, len(y_pred_np), num_nodes, 1) * sigma_tensor
y_ensemble_t = y_pred_t.unsqueeze(0) + noise

print("\n--- Evaluation Metrics (Aggregated Across All Nodes) ---")
print(f"{mae(y_true_t, y_pred_t):.4f}")
print(f"{mase(y_true_t, y_pred_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_t, y_train_t):.4f}")
print(f"{empirical_coverage(y_true_np, lower_bounds_k15_np, upper_bounds_k15_np):.4f}")
print(f"{crps_approximation(y_true_t, y_ensemble_t):.4f}")
print(f"{winkler_score(y_true_np, lower_bounds_95_np, upper_bounds_95_np, alpha_winkler):.4f}")
print(f"{pinball_loss(y_true_t, pred_80_t, q_80):.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")


# Plotting results for first 3 nodes
plot_nodes = min(3, num_nodes)
fig, axes = plt.subplots(plot_nodes, 1, figsize=(12, 4 * plot_nodes), sharex=True)

if plot_nodes == 1:
    axes = [axes]

time_index = df.index[-horizon:]
train_index_plot = df.index[-(horizon + 48):-horizon]

for i, ax in enumerate(axes):
    node_name = df.columns[i]
    
    ax.plot(train_index_plot, y_train_np[-48:, i], label="Train", color="black")
    ax.plot(time_index, y_true_np[:, i], label="Ground Truth", color="blue", marker="o", markersize=4)
    ax.plot(time_index, y_pred_np[:, i], label="Point Forecast", color="red", linestyle="--", marker="x", markersize=4)
    
    ax.fill_between(
        time_index,
        lower_bounds_k15_np[:, i],
        upper_bounds_k15_np[:, i],
        color="red",
        alpha=0.2,
        label="Prediction Interval (k=1.5)"
    )
    
    ax.set_title(f"Forecast vs Actuals: {node_name}")
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc="upper left")

plt.xlabel("Time Step")
plt.tight_layout()
plt.show()

# 2. Resampling-based Ensemble: LSTM-BE (Bootstrap Ensemble)

In [ ]:
df = pd.read_csv(data_path)
num_nodes = df.shape[1]
series = TimeSeries.from_dataframe(df)

horizon = 30
train, val = series[:-horizon], series[-horizon:]

# Standardize
scaler = Scaler()
train_scaled = scaler.fit_transform(train)

# Model training
print(f"Training LSTM model on shape {df.shape} with standardized data...")
model = BlockRNNModel(
    model="LSTM",
    input_chunk_length=60,
    hidden_dim=64,
    n_rnn_layers=2,
    output_chunk_length=horizon,
    n_epochs=100,
    random_state=42,
    pl_trainer_kwargs={"enable_progress_bar": True, "logger": False}
)

start_train = time.time()
model.fit(train_scaled)
training_time = time.time() - start_train

print("Generating in-sample predictions for residuals...")
in_sample_preds_scaled = model.historical_forecasts(
    train_scaled,
    start=model.input_chunk_length,
    forecast_horizon=1,
    retrain=False,
    verbose=False
)

# Invert transform to calculate residuals on the original scale
in_sample_preds = scaler.inverse_transform(in_sample_preds_scaled)
in_sample_true = train.slice_intersect(in_sample_preds)

# Store full empirical residual matrix: Shape (T_hist, num_nodes)
residuals = in_sample_true.values() - in_sample_preds.values()

print("Generating out-of-sample forecast...")
start_infer = time.time()
point_forecast_scaled = model.predict(n=horizon)
inference_time = time.time() - start_infer

# Invert transform for final predictions
point_forecast = scaler.inverse_transform(point_forecast_scaled)

y_pred_np = point_forecast.values()
y_true_np = val.values()
y_train_np = train.values() # Keep original scale for baseline metric calculations

# Residual Bootstrap Ensemble Methodology
np.random.seed(42)
M = 100  # Number of bootstrap samples in ensemble
T_hist = residuals.shape[0]

# Sample random time indices to pull residual vectors.
# By sampling row indices, we preserve the spatial correlation across nodes at time 't'.
sampled_indices = np.random.randint(0, T_hist, size=(M, horizon))

# Extract the resampled residuals: Shape (M, horizon, num_nodes)
resampled_residuals = residuals[sampled_indices]

# Create ensemble of trajectories: y_hat + e_bootstrap
# Broadcast y_pred_np (horizon, num_nodes) -> (M, horizon, num_nodes)
y_ensemble_np = y_pred_np[np.newaxis, :, :] + resampled_residuals

# Extract empirical quantiles from the ensemble
# For Winkler Score at alpha = 0.05 (95% CI)
alpha_winkler = 0.05
lower_bounds_95_np = np.quantile(y_ensemble_np, alpha_winkler/2, axis=0)
upper_bounds_95_np = np.quantile(y_ensemble_np, 1 - alpha_winkler/2, axis=0)

# For Pinball at 0.8 quantile
q_80 = 0.8
pred_80_np = np.quantile(y_ensemble_np, q_80, axis=0)

# Metrics
def format_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32).unsqueeze(-1)

y_true_t = format_tensor(y_true_np)
y_pred_t = format_tensor(y_pred_np)
y_train_t = format_tensor(y_train_np)
pred_80_t = format_tensor(pred_80_np)

# Convert ensemble to torch tensor for CRPS: Shape (M, horizon, num_nodes, 1)
y_ensemble_t = torch.tensor(y_ensemble_np, dtype=torch.float32).unsqueeze(-1)

print("\n--- Evaluation Metrics (Aggregated Across All Nodes) ---")
print(f"{mae(y_true_t, y_pred_t):.4f}")
print(f"{mase(y_true_t, y_pred_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_t, y_train_t):.4f}")
print(f"{empirical_coverage(y_true_np, lower_bounds_95_np, upper_bounds_95_np):.4f}")
print(f"{crps_approximation(y_true_t, y_ensemble_t):.4f}")
print(f"{winkler_score(y_true_np, lower_bounds_95_np, upper_bounds_95_np, alpha_winkler):.4f}")
print(f"{pinball_loss(y_true_t, pred_80_t, q_80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")


# Plotting (first 3 nodes)
plot_nodes = min(3, num_nodes)
fig, axes = plt.subplots(plot_nodes, 1, figsize=(12, 4 * plot_nodes), sharex=True)

if plot_nodes == 1:
    axes = [axes]

time_index = df.index[-horizon:]
train_index_plot = df.index[-(horizon + 48):-horizon]

for i, ax in enumerate(axes):
    node_name = df.columns[i]
    
    ax.plot(train_index_plot, y_train_np[-48:, i], label="Train", color="black")
    ax.plot(time_index, y_true_np[:, i], label="Ground Truth", color="blue", marker="o", markersize=4)
    ax.plot(time_index, y_pred_np[:, i], label="Point Forecast", color="red", linestyle="--", marker="x", markersize=4)
    
    ax.fill_between(
        time_index,
        lower_bounds_95_np[:, i],
        upper_bounds_95_np[:, i],
        color="red",
        alpha=0.2,
        label="Prediction Interval (Empirical 95%)"
    )
    
    ax.set_title(f"Forecast vs Actuals (Bootstrap): {node_name}")
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc="upper left")

plt.xlabel("Time Step")
plt.tight_layout()
plt.show()

# 3. Conformal Prediction: NHITS-CP

In [ ]:
df = pd.read_csv(data_path)
num_nodes = df.shape[1]
series = TimeSeries.from_dataframe(df)

horizon = 30
train, val = series[:-horizon], series[-horizon:]
alpha_winkler = 0.05

# Split the dataset: test sequence (val) and everything prior (train_full)
train_val_split_idx = -horizon
train_full, val = series[:train_val_split_idx], series[train_val_split_idx:]

# Conformal prediction requires a calibration set to extract historical errors.
# We reserve the last 200 points of `train_full` for calibration. 
cal_len = max(200, 2 * horizon)
base_train, cal = train_full[:-cal_len], train_full[-cal_len:]


# Model training and conformal wrapper
print(f"Training base NHITS model on shape {base_train.to_dataframe().shape}...")
start_train = time.time()
base_model = NHiTSModel(
    input_chunk_length=60,
    output_chunk_length=horizon,
    n_epochs=100,
    random_state=42,
    pl_trainer_kwargs={"enable_progress_bar": True, "logger": False}
)
base_model.fit(base_train)

print("Fitting ConformalNaiveModel on calibration set...")
# Target quantiles for median, Pinball-80, and Winkler-95 bounds
quantiles = [0.025, 0.20, 0.50, 0.80, 0.975]
cp_model = ConformalNaiveModel(model=base_model, quantiles=quantiles)
cp_model.fit(series=cal)
training_time = time.time() - start_train

print("Generating out-of-sample probabilistic forecast...")
n_samples = 100
start_time = time.time()
pred = cp_model.predict(n=horizon, series=train_full, num_samples=n_samples)
inference_time = time.time() - start_time
y_pred_np = pred.quantile(0.50).values()
lower_bounds_95_np = pred.quantile(0.025).values()
upper_bounds_95_np = pred.quantile(0.975).values()
pred_80_np = pred.quantile(0.80).values()

# Format the ensemble for CRPS. Darts outputs shape: (time, nodes, samples)
y_ensemble_np = pred.all_values().transpose(2, 0, 1)[:, :, :, np.newaxis]
print(y_ensemble_np.shape)

y_true_np = val.values()
y_train_np = train_full.values()

# Metrics
def format_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32).unsqueeze(-1)

y_true_t = format_tensor(y_true_np)
y_pred_t = format_tensor(y_pred_np)
y_train_t = format_tensor(y_train_np)
pred_80_t = format_tensor(pred_80_np)
y_ensemble_t = torch.tensor(y_ensemble_np, dtype=torch.float32)

print("\n--- Evaluation Metrics (Aggregated Across All Nodes) ---")
print(f"{mae(y_true_t, y_pred_t):.4f}")
print(f"{mase(y_true_t, y_pred_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_t, y_train_t):.4f}")
print(f"{empirical_coverage(y_true_np, lower_bounds_95_np, upper_bounds_95_np):.4f}")
print(f"{crps_approximation(y_true_t, y_ensemble_t):.4f}")
print(f"{winkler_score(y_true_np, lower_bounds_95_np, upper_bounds_95_np, alpha_winkler):.4f}")
print(f"{pinball_loss(y_true_t, pred_80_t, 0.80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 4. Conformal Quantile Regression: NHITS-CQR

In [ ]:
df = pd.read_csv(data_path)
num_nodes = df.shape[1]
series = TimeSeries.from_dataframe(df)

horizon = 30
q_80 = 0.8
P_LAG = 60
T_PRED = horizon

# Split the dataset: test sequence (val) and everything prior (train_full)
train_val_split_idx = -horizon
train_full, val = series[:train_val_split_idx], series[train_val_split_idx:]

# Split train_full into train_base (for the base model) and calib (for ConformalQRModel).
calib_size = max(90, 3 * horizon) 
train_base = train_full[:-calib_size]
calib = train_full[-calib_size:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Scale the data based strictly on train_base to prevent data leakage
scaler = Scaler()
train_base_scaled = scaler.fit_transform(train_base)
calib_scaled = scaler.transform(calib)
train_full_scaled = scaler.transform(train_full)

# Extract raw NumPy arrays for metric evaluations on the original scale
train_values = train_full.to_dataframe().values
val_values = val.to_dataframe().values
quantiles = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5, 0.8, 0.9, 0.95, 0.975, 0.99]

print("Training base NHITS Model...")
start_train = time.time()
base_model = NHiTSModel(
    input_chunk_length=P_LAG,
    output_chunk_length=T_PRED,
    dropout=0.1,
    batch_size=64,
    n_epochs=100,
    optimizer_kwargs={"lr": 1e-3},
    random_state=42,
    pl_trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu"}
)
base_model.fit(train_base_scaled, verbose=True)

print("Fitting ConformalQRModel on the calibration set...")
conformal_model = ConformalQRModel(model=base_model, quantiles=quantiles)

# Fit conformal prediction intervals using the holdout calibration series
conformal_model.fit(series=calib_scaled)
training_time = time.time() - start_train

print("Generating calibrated probabilistic forecasts...")
num_samples = 100
start_time = time.time()
forecast_scaled = conformal_model.predict(n=horizon, series=train_full_scaled, num_samples=num_samples)

# Inverse transform to get predictions back to original scale
forecast = scaler.inverse_transform(forecast_scaled)

# Extract and transpose from Darts format (time, components, samples) to (samples, time, components)
forecast_values = forecast.all_values() 
y_preds_ensemble = np.transpose(forecast_values, (2, 0, 1))
y_pred_mean = np.mean(y_preds_ensemble, axis=0)

alpha = 0.05
lower_bounds = np.percentile(y_preds_ensemble, (alpha/2)*100, axis=0)
upper_bounds = np.percentile(y_preds_ensemble, (1 - alpha/2)*100, axis=0)
y_pred_80 = np.percentile(y_preds_ensemble, q_80*100, axis=0)
inference_time = time.time() - start_time

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_pred_mean_t = torch.tensor(y_pred_mean, dtype=torch.float32).to(device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)
y_pred_80_t = torch.tensor(y_pred_80, dtype=torch.float32).to(device)
emp_cov = empirical_coverage(val_values, lower_bounds, upper_bounds)
wink_score = winkler_score(val_values, lower_bounds, upper_bounds, alpha)

print("\n--- Metrics ---")
print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80):.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 5. Bayesian Neural Network - Monte Carlo Dropout (BNN-MCD)

In [ ]:
# Set random seed for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df = pd.read_csv(data_path)
num_nodes = df.shape[1]
series = TimeSeries.from_dataframe(df)

horizon = 30
train, val = series[:-horizon], series[-horizon:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract numpy arrays for PyTorch
train_values = train.to_dataframe().values
val_values = val.to_dataframe().values
num_nodes = train_values.shape[1]

# Scale the data 
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_values)
val_scaled = scaler.transform(val_values)
seq_len = 60  # Look-back window 
horizon = 30  # Forecast horizon

def create_sequences(data, seq_len, horizon):
    X, y = [], []
    for i in range(len(data) - seq_len - horizon + 1):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len : i + seq_len + horizon])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

X_train, y_train_seq = create_sequences(train_scaled, seq_len, horizon)

train_dataset = TensorDataset(X_train, y_train_seq)

# Add a PyTorch generator to ensure DataLoader shuffling is reproducible
g = torch.Generator()
g.manual_seed(42)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)

# Define the BNN-MCD with LSTM as base model
class BNN_MCDropout(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64, dropout_rate=0.3):
        super(BNN_MCDropout, self).__init__()
        self.num_nodes = num_nodes
        self.horizon = horizon
        
        # LSTM with dropout on the outputs of each RNN layer
        self.lstm = nn.LSTM(
            input_size=num_nodes, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            dropout=dropout_rate
        )
        # Standard dropout layer for the dense network
        self.dropout = nn.Dropout(dropout_rate)
        # Output layer maps hidden state to a flattened prediction of (horizon * num_nodes)
        self.fc = nn.Linear(hidden_dim, horizon * num_nodes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]          # Take the last sequence output
        out = self.dropout(out)      # Apply MC Dropout
        out = self.fc(out)
        return out.view(-1, self.horizon, self.num_nodes)

model = BNN_MCDropout(num_nodes=num_nodes).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Model training
epochs = 100
model.train() # Ensures dropout is active
start_time = time.time()
print("Training BNN...")
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

training_time = time.time() - start_time

# Inference
X_test = torch.tensor(train_scaled[-seq_len:], dtype=torch.float32).unsqueeze(0).to(device)

start_time = time.time()
model.train() # Keep dropout for Monte Carlo sampling!
num_samples = 100
y_preds_ensemble_scaled = []

print("Sampling from posterior...")
with torch.no_grad():
    for _ in range(num_samples):
        # Shape: (1, horizon, num_nodes)
        pred = model(X_test).squeeze(0).cpu().numpy() 
        y_preds_ensemble_scaled.append(pred)

y_preds_ensemble_scaled = np.array(y_preds_ensemble_scaled)

# Inverse transform predictions back to original scale
y_preds_ensemble = np.zeros_like(y_preds_ensemble_scaled)
for i in range(num_samples):
    y_preds_ensemble[i] = scaler.inverse_transform(y_preds_ensemble_scaled[i])

y_pred_mean = np.mean(y_preds_ensemble, axis=0)

inference_time = time.time() - start_time

# Define alpha for 95% Confidence Intervals (used for Winkler Score)
alpha = 0.05
quantile_lower, quantile_upper = (alpha/2)*100, (1 - alpha/2)*100
lower_bounds = np.percentile(y_preds_ensemble, quantile_lower, axis=0)
upper_bounds = np.percentile(y_preds_ensemble, quantile_upper, axis=0)

# Extract the 80th quantile specifically for Pinball Loss
y_pred_80 = np.percentile(y_preds_ensemble, 80, axis=0) 

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_pred_mean_t = torch.tensor(y_pred_mean, dtype=torch.float32).to(device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)

# Convert the 80th quantile array to a tensor
y_pred_80_t = torch.tensor(y_pred_80, dtype=torch.float32).to(device)

print("\n--- Evaluation Metrics ---")
print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
emp_cov = empirical_coverage(val_values, lower_bounds, upper_bounds)
wink_score = winkler_score(val_values, lower_bounds, upper_bounds, alpha)
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
# Calculate Pinball loss at the 80th quantile
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=0.80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 6. Fast Gaussian Process (GpGp)

In [ ]:
location_path = 'location_coordinates.csv' # CSV file containing location information as [name, longitude, latitude]
horizon = 30
num_samples = 100
prediction_output_file = 'GpGp_predictions.csv'
simulations_output_file = 'GpGp_sims.csv'

# We use gpgp library from R
r_script_content = f"""
# Safely install missing dependencies first
if (!require("fields", character.only = TRUE)) {{
  install.packages("fields", repos="http://cran.us.r-project.org")
}}
if (!require("GpGp", character.only = TRUE)) {{
  install.packages("GpGp", repos="http://cran.us.r-project.org")
  library(GpGp)
}}

# Load data
data_path <- '{data_path}'
location_path <- '{location_path}'
T_PRED <- {horizon}
NSIM <- {num_samples}

# Read the main data (drop index/time column)
data <- read.csv(data_path, fileEncoding = "latin1")
data <- data[, -1] 

# Delete last T_PRED rows in training data
data <- data[1:(nrow(data) - T_PRED), ]

# Read location coordinates
location_data <- read.csv(location_path, fileEncoding = "latin1")
lon_lat <- as.matrix(location_data[, 2:3])

nlocs <- ncol(data)
ntimes <- nrow(data)

# Create spatio-temporal location matrix (time changes slowest, loc changes fastest)
locs <- do.call(rbind, replicate(ntimes, lon_lat, simplify = FALSE))
time_col <- rep(1:ntimes, each = nlocs)
locs <- cbind(locs, time = time_col)

# Vectorize the response matrix
y <- as.vector(t(data))

# Track Training Time
start_train <- Sys.time()

# Fit the GpGp model
fit <- fit_model(
  y = y,
  locs = locs,
  covfun_name = "matern_isotropic", silent=TRUE
)

end_train <- Sys.time()
train_time <- as.numeric(difftime(end_train, start_train, units = "secs"))

# Prediction setup
new_time <- (ntimes+1):(ntimes+T_PRED)
new_locs <- do.call(rbind, replicate(T_PRED, lon_lat, simplify = FALSE))
new_time_col <- rep(new_time, each = nlocs)
new_locs <- cbind(new_locs, time = new_time_col)
X_pred <- matrix(1, nrow = nrow(new_locs), ncol = 1)

# Track Inference Time
start_infer <- Sys.time()

# Predict (Mean)
pred <- predictions(
  fit = fit,
  locs_pred = as.matrix(new_locs),
  X_pred = X_pred
)

# Predict (Conditional Simulations for Probabilistic Ensemble)
sims <- cond_sim(
  fit = fit, 
  locs_pred = as.matrix(new_locs), 
  X_pred = X_pred, 
  nsim = NSIM
)

end_infer <- Sys.time()
infer_time <- as.numeric(difftime(end_infer, start_infer, units = "secs"))

# Save the predictions and tracked times to CSV for Python to load
out_df <- data.frame(
  Prediction = pred, 
  TrainTime = train_time, 
  InferTime = infer_time
)
write.csv(out_df, '{prediction_output_file}', row.names=FALSE)

# Save the simulations matrix
write.csv(sims, '{simulations_output_file}', row.names=FALSE)
"""

# Write script to disk
with open('run_gpgp.R', 'w') as file:
    file.write(r_script_content)

# Run the R script
print("Running R script...")
try:
    result = subprocess.run(['Rscript', 'run_gpgp.R'], capture_output=True, text=True, check=True)
    print("R script finished successfully.")
except subprocess.CalledProcessError as e:
    print("FAILED! R returned the following error:\n", "-"*40)
    print(e.stderr)
    print("-" * 40)
    raise

# Load Predictions and Data in Python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Original Full Data to extract ground truth
df = pd.read_csv(data_path)
train_values = df.iloc[:-horizon].values
val_values = df.iloc[-horizon:].values
num_nodes = df.shape[1]

# Load Mean Predictions
pred_df = pd.read_csv(prediction_output_file)
y_pred_mean = pred_df['Prediction'].values.reshape(horizon, num_nodes)
training_time = pred_df['TrainTime'].iloc[0]
inference_time = pred_df['InferTime'].iloc[0]

# Load and Reshape Conditional Simulations
# R outputs simulations of shape (T*N, M)
sims_df = pd.read_csv(simulations_output_file)
samples_np = sims_df.values 

# Pivot to (M, T, N). Python 'C' order reshapes perfectly align with the R location matrix construction 
# where time changes slowest and nodes change fastest.
y_preds_ensemble = samples_np.T.reshape(num_samples, horizon, num_nodes)

# Calculate percentiles based on the true model ensemble
alpha = 0.05
lower_bounds = np.percentile(y_preds_ensemble, (alpha/2)*100, axis=0)
upper_bounds = np.percentile(y_preds_ensemble, (1 - alpha/2)*100, axis=0)
y_pred_80 = np.percentile(y_preds_ensemble, 80, axis=0)

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_pred_mean_t = torch.tensor(y_pred_mean, dtype=torch.float32).to(device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)
y_pred_80_t = torch.tensor(y_pred_80, dtype=torch.float32).to(device)
emp_cov = empirical_coverage(val_values, lower_bounds, upper_bounds)
wink_score = winkler_score(val_values, lower_bounds, upper_bounds, alpha)

print("\n--- Evaluation Metrics ---")
print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=0.80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 7. Parametric Predictive Distribution: DeepAR

In [ ]:
df = pd.read_csv(data_path)
num_nodes = df.shape[1]
series = TimeSeries.from_dataframe(df)
horizon = 30
P_LAG = 60
T_PRED = horizon

train, val = series[:-horizon], series[-horizon:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

scaler = Scaler()
train_scaled = scaler.fit_transform(train)

# Extract raw NumPy arrays for metrics later
train_values = train.to_dataframe().values
val_values = val.to_dataframe().values

# Model training
print("Training DeepAR (RNN with Gaussian Likelihood)...")
start_time = time.time()

model = RNNModel(
    model="RNN",
    hidden_dim=64,
    n_rnn_layers=2,
    dropout=0.1,
    batch_size=64,
    n_epochs=100,
    optimizer_kwargs={"lr": 1e-3},
    random_state=42,
    input_chunk_length=P_LAG,
    training_length=P_LAG + T_PRED,
    likelihood=GaussianLikelihood(),
    pl_trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu"}
)

model.fit(train_scaled, verbose=True)
training_time = time.time() - start_time

print("Generating probabilistic forecasts...")
num_samples = 100
start_time = time.time()
forecast_scaled = model.predict(n=horizon, series=train_scaled, num_samples=num_samples)

# Inverse transform (Darts applies this to all trajectories automatically)
forecast = scaler.inverse_transform(forecast_scaled)

# Darts stores samples as (time, components, samples). Extract and transpose to (samples, time, components)
forecast_values = forecast.all_values() 
y_preds_ensemble = np.transpose(forecast_values, (2, 0, 1))

y_pred_mean = np.mean(y_preds_ensemble, axis=0)
inference_time = time.time() - start_time
alpha = 0.05
lower_bounds = np.percentile(y_preds_ensemble, (alpha/2)*100, axis=0)
upper_bounds = np.percentile(y_preds_ensemble, (1 - alpha/2)*100, axis=0)
y_pred_80 = np.percentile(y_preds_ensemble, 80, axis=0)

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_pred_mean_t = torch.tensor(y_pred_mean, dtype=torch.float32).to(device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)
y_pred_80_t = torch.tensor(y_pred_80, dtype=torch.float32).to(device)
emp_cov = empirical_coverage(val_values, lower_bounds, upper_bounds)
wink_score = winkler_score(val_values, lower_bounds, upper_bounds, alpha)

print("\n--- Metrics ---")
print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=0.80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 8. DeepNPTS

In [ ]:
# Force spawn method for PyTorch multiprocessing to avoid CUDA re-initialization errors
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

df = pd.read_csv(data_path)

horizon = 30
num_nodes = df.shape[1]
q_80 = 0.8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract arrays for evaluation (shape: time, num_nodes)
train_values = df.iloc[:-horizon].values
val_values = df.iloc[-horizon:].values
node_names = df.columns.tolist()

# NeuralForecast requires 'unique_id', 'ds', and 'y' columns.
df['ds'] = pd.date_range(start='2020-01-01', periods=len(df))
long_df = df.melt(id_vars=['ds'], var_name='unique_id', value_name='y')

# Split train set in long format
train_df = long_df.groupby('unique_id').apply(lambda x: x.iloc[:-horizon]).reset_index(drop=True)

print(f"Training DeepNPTS model on {num_nodes} nodes...")
model = DeepNPTS(
    h=horizon,
    input_size=60,
    scaler_type='standard',
    random_seed=42,
    max_steps=100,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1
)

nf = NeuralForecast(models=[model], freq='D')

start_train = time.time()
nf.fit(
    df=train_df, 
    val_size=0, 
    prediction_intervals=PredictionIntervals(n_windows=3) 
)
training_time = time.time() - start_train

print("Generating out-of-sample probabilistic forecast...")
start_infer = time.time()
forecasts = nf.predict(df=train_df, level=[60, 95])
inference_time = time.time() - start_infer

# Convert back to wide format (horizon, num_nodes)
y_pred_np = np.zeros((horizon, num_nodes))
lower_bounds_np = np.zeros((horizon, num_nodes))
upper_bounds_np = np.zeros((horizon, num_nodes))
y_pred_80_np = np.zeros((horizon, num_nodes))

# Ensure ordering matches original columns
for i, node in enumerate(node_names):
    node_fcst = forecasts[forecasts.index == node] if forecasts.index.name == 'unique_id' else forecasts[forecasts['unique_id'] == node]
    node_fcst = node_fcst.sort_values(by='ds')
    
    y_pred_np[:, i] = node_fcst['DeepNPTS'].values
    lower_bounds_np[:, i] = node_fcst['DeepNPTS-lo-95'].values
    upper_bounds_np[:, i] = node_fcst['DeepNPTS-hi-95'].values
    y_pred_80_np[:, i] = node_fcst['DeepNPTS-hi-60'].values  # 80th percentile

# Approximate sigma_hat from 95% intervals for drawing an ensemble
z_975 = norm.ppf(0.975)
sigma_hat = (upper_bounds_np - lower_bounds_np) / (2 * z_975)

# Simulate 100-sample ensemble for CRPS approximation
np.random.seed(42)
num_samples = 100
y_preds_ensemble = np.random.normal(loc=y_pred_np, scale=sigma_hat, size=(num_samples, horizon, num_nodes))


# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32).unsqueeze(-1).to(device)
y_pred_mean_t = torch.tensor(y_pred_np, dtype=torch.float32).unsqueeze(-1).to(device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble, dtype=torch.float32).unsqueeze(-1).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).unsqueeze(-1).to(device)
y_pred_80_t = torch.tensor(y_pred_80_np, dtype=torch.float32).unsqueeze(-1).to(device)

alpha_winkler = 0.05
emp_cov = empirical_coverage(val_values, lower_bounds_np, upper_bounds_np)
wink_score = winkler_score(val_values, lower_bounds_np, upper_bounds_np, alpha_winkler)

print("\n--- Metrics ---")
print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 9. Quantile Regression: LSTM-QR

In [ ]:
df = pd.read_csv(data_path)
num_nodes = df.shape[1]
series = TimeSeries.from_dataframe(df)

horizon = 30
P_LAG = 60
T_PRED = horizon
q_80 = 0.8

train, val = series[:-horizon], series[-horizon:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Scale the data
scaler = Scaler()
train_scaled = scaler.fit_transform(train)

# Extract raw NumPy arrays for metrics evaluation (on original scale)
train_values = train.to_dataframe().values
val_values = val.to_dataframe().values

# Model Setup & Training (LSTM + QuantileRegression)
# Comprehensive list of quantiles for the likelihood to fit
quantiles = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5, 0.8, 0.9, 0.95, 0.975, 0.99]

model = BlockRNNModel(
    model="LSTM",
    input_chunk_length=P_LAG,
    output_chunk_length=T_PRED,
    hidden_dim=64,
    n_rnn_layers=2,
    dropout=0.1,
    batch_size=64,
    n_epochs=100,
    optimizer_kwargs={"lr": 1e-3},
    random_state=42,
    likelihood=QuantileRegression(quantiles=quantiles),
    pl_trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu"}
)

# Fit on scaled data
start_train = time.time()
model.fit(train_scaled, verbose=True)
training_time = time.time() - start_train

num_samples = 100

# Predict on scaled data and track inference time
start_infer = time.time()
forecast_scaled = model.predict(n=horizon, series=train_scaled, num_samples=num_samples)
inference_time = time.time() - start_infer

# Inverse transform to get back to original scale
forecast = scaler.inverse_transform(forecast_scaled)

# Darts stores samples as (time, components, samples). Extract and transpose to (samples, time, components)
forecast_values = forecast.all_values() 
y_preds_ensemble = np.transpose(forecast_values, (2, 0, 1))

y_pred_mean = np.mean(y_preds_ensemble, axis=0)
alpha = 0.05
lower_bounds = np.percentile(y_preds_ensemble, (alpha/2)*100, axis=0)
upper_bounds = np.percentile(y_preds_ensemble, (1 - alpha/2)*100, axis=0)
y_pred_80 = np.percentile(y_preds_ensemble, q_80*100, axis=0)

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_pred_mean_t = torch.tensor(y_pred_mean, dtype=torch.float32).to(device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)
y_pred_80_t = torch.tensor(y_pred_80, dtype=torch.float32).to(device)
emp_cov = empirical_coverage(val_values, lower_bounds, upper_bounds)
wink_score = winkler_score(val_values, lower_bounds, upper_bounds, alpha)

print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80):.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 10. Engression: GCEN

In [ ]:
# Load the matrix containing Haversine distances for each node and compute the adjacency matrix
# Hyperparameters are taken from: https://stengression.readthedocs.io/examples/Spatiotemporal_Engression_Examples.html
distances = pd.read_csv("distances_path.csv")
dist_array = distances.values
adjacency_matrix = compute_adjacency_matrix(distances.values, sigma2=0.9948808074595779, epsilon=0.07692416606625796, n=40)
np.fill_diagonal(adjacency_matrix, 1)

node_indices, neighbor_indices = np.where(adjacency_matrix == 1)
graph_info = GraphInfo(
    edges=(node_indices.tolist(), neighbor_indices.tolist()),
    num_nodes=adjacency_matrix.shape[0],
)
print(f"Number of nodes: {graph_info.num_nodes}, Number of edges: {len(graph_info.edges[0])}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv(data_path)
NUM_NODES = df.shape[1]
IN_FEAT_DIM = 1   # D

P_LAG = 60        # Lag window, input_seq_len
T_PRED = 30       # Prediction horizon, output_seq_len
BATCH_SIZE = 64

sts_data = torch.tensor(df.values, dtype=torch.float32)
sts_data = sts_data.view(sts_data.shape[0], sts_data.shape[1], 1) 
sts_data = sts_data.to(device)
node_names = df.columns.tolist()

train_tensor = sts_data[:-T_PRED, :, :]
test_ground_truth = sts_data[-T_PRED:, :, :] 
test_history = train_tensor[-P_LAG:, :, :] 

# Extract validation and train values for metrics downstream
val_values = test_ground_truth.squeeze(-1).cpu().numpy()
train_values = train_tensor.squeeze(-1).cpu().numpy()

# Standardize train data
y_train = train_tensor.clone()
mean = train_tensor.mean(dim=0, keepdim=True)
std = train_tensor.std(dim=0, keepdim=True)
std[std == 0] = 1e-8  # Prevent division by zero

train_tensor = ((train_tensor - mean) / std)
test_history = ((test_history - mean) / std) 

train_dataset = SpatioTemporalDataset(train_tensor, input_seq_len=P_LAG, output_seq_len=T_PRED, multi_horizon=True)
trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

# We take the hyperparameters from https://stengression.readthedocs.io/en/latest/examples/Spatiotemporal_Engression_Examples.html
gcen = GCEN(in_feat_dim=1,
    gcn_out_feat=6,
    lstm_hidden_dim=91,
    lstm_num_layers=5,
    lstm_dropout=0.30479293048737066,
    p_lag=P_LAG,
    t_pred=T_PRED,
    graph_info=graph_info,
    noise_encode="add",
    noise_dist="uniform",
    gcn_seed=21,
    temporal_seed=9
).to(device)
optimizer = optim.Adam(gcen.parameters(), lr=0.00426979947571805) 

# Load pre-trained weights from https://github.com/PyCoder913/stengression/tree/main/examples
gcen.load_state_dict(torch.load('GCEN_Belgium_30_Weights.pth'))

start_time = time.time()
forecast_ensemble = gcen.predict(m_samples=100, history=test_history, unstandardize=[mean, std], device=device) # (M, T, N, 1)
inference_time = time.time() - start_time
forecast_ensemble = forecast_ensemble.float().to(device)

y_preds_ensemble_t = torch.squeeze(forecast_ensemble, dim=-1)
y_pred_median_t = torch.median(y_preds_ensemble_t, dim=0).values
alpha = 0.05
q_80 = 0.8
lower_bounds_t = torch.quantile(y_preds_ensemble_t, q=alpha / 2, dim=0)
upper_bounds_t = torch.quantile(y_preds_ensemble_t, q=1 - alpha / 2, dim=0)
y_pred_80_t = torch.quantile(y_preds_ensemble_t, q=q_80, dim=0)
lower_bounds_np = lower_bounds_t.cpu().numpy()
upper_bounds_np = upper_bounds_t.cpu().numpy()

emp_cov = empirical_coverage(val_values, lower_bounds_np, upper_bounds_np)
wink_score = winkler_score(val_values, lower_bounds_np, upper_bounds_np, alpha)
y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)

# Metrics
print(f"{mae(y_true_t, y_pred_median_t):.4f}")
print(f"{mase(y_true_t, y_pred_median_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_median_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80):.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 11. K2VAE

In [ ]:
# Import dependencies and define the model
!git clone https://github.com/decisionintelligence/K2VAE.git
sys.path.append('../K2VAE')
!pip install lightning reformer_pytorch linear_attention_transformer
from probts.model.nn.prob.k2VAE.k2vae import k2VAE
import torch
from einops import rearrange
from sympy.stats.rv import sampling_E
from probts.model.nn.prob.k2VAE.k2vae import k2VAE
import torch.nn as nn
import torch.nn.functional as F

import torch
from torch import nn
from typing import List

from probts.utils import weighted_average
from probts.data.data_utils.data_scaler import TemporalScaler
from typing import Union

class Forecaster(nn.Module):
    def __init__(
        self,
        target_dim: int,
        context_length: Union[list,int],
        prediction_length: Union[list,int],
        freq: str ,
        use_lags: bool = False,
        use_feat_idx_emb: bool = False,
        use_time_feat: bool = False,
        lags_list: List[int] = None,
        feat_idx_emb_dim: int = 1,
        time_feat_dim: int = 1,
        use_scaling: bool = False,
        autoregressive: bool = False,
        no_training: bool = False,
        dataset: str = None,
        **kwargs
    ):
        super().__init__()
        
        self.context_length = context_length
        self.prediction_length = prediction_length
        
        if isinstance(self.context_length, list):
            self.max_context_length = max(self.context_length)
        else:
            self.max_context_length = self.context_length
        
        if isinstance(self.prediction_length, list):
            self.max_prediction_length = max(self.prediction_length)
        else:
            self.max_prediction_length = self.prediction_length
            
        self.target_dim = target_dim
        self.freq = freq
        self.use_lags = use_lags
        self.use_feat_idx_emb = use_feat_idx_emb
        self.use_time_feat = use_time_feat
        self.feat_idx_emb_dim = feat_idx_emb_dim
        self.time_feat_dim = time_feat_dim
        self.autoregressive = autoregressive
        self.no_training = no_training
        self.use_scaling = use_scaling
        self.dataset = dataset
        # Lag parameters
        self.lags_list = lags_list
        if self.use_scaling:
            self.scaler = TemporalScaler()
        else:
            self.scaler = None

        if self.lags_list is not None:
            self.lags_dim = len(self.lags_list) * target_dim
        
        if use_feat_idx_emb:
            self.feat_idx_emb = nn.Embedding(
                num_embeddings=self.target_dim, embedding_dim=self.feat_idx_emb_dim
            )
        else:
            self.feat_idx_emb = None
            
        self.input_size = self.get_input_size()
            

    @property
    def name(self):
        return self.__class__.__name__

    def get_input_size(self):
        input_size = self.target_dim if not self.use_lags else self.lags_dim
        if self.use_feat_idx_emb:
            input_size += self.use_feat_idx_emb * self.target_dim
        if self.use_time_feat:
            input_size += self.time_feat_dim
        return input_size

    def get_lags(self, sequence, lags_list, lags_length=1):
        """
        Get several lags from the sequence of shape (B, L, C) to (B, L', C*N),
        where L' = lag_length and N = len(lag_list).
        """
        assert max(lags_list) + lags_length <= sequence.shape[1]

        lagged_values = []
        for lag_index in lags_list:
            begin_index = -lag_index - lags_length
            end_index = -lag_index if lag_index > 0 else None
            lagged_value = sequence[:, begin_index:end_index, ...]
            if self.use_scaling:
                lagged_value = lagged_value / self.scaler.scale
            lagged_values.append(lagged_value)
        return torch.cat(lagged_values, dim=-1)

    def get_input_sequence(
        self,
        past_target_cdf,
        future_target_cdf,
        mode
    ):
        if mode == 'all':
            sequence = torch.cat((past_target_cdf, future_target_cdf), dim=1)
            seq_length = self.max_context_length + self.max_prediction_length
        elif mode == 'encode':
            sequence = past_target_cdf
            seq_length = self.max_context_length
        elif mode == 'decode':
            sequence = past_target_cdf
            seq_length = 1
        else:
            raise ValueError(f"Unsupported input mode: {mode}")
        
        if self.use_lags:
            input_seq = self.get_lags(sequence, self.lags_list, seq_length)
        else: 
            input_seq = sequence[:, -seq_length:, ...]
            if self.use_scaling:
                input_seq = input_seq / self.scaler.scale
        return input_seq
    
    def get_input_feat_idx_emb(self, target_dimension_indicator, input_length):
        input_feat_idx_emb = self.feat_idx_emb(target_dimension_indicator) # [B K D]

        input_feat_idx_emb = (
            input_feat_idx_emb.unsqueeze(1)
            .expand(-1, input_length, -1, -1)
            .reshape(-1, input_length, self.target_dim * self.feat_idx_emb_dim)
        )
        return input_feat_idx_emb # [B L K*D]

    def get_input_time_feat(
        self,
        past_time_feat,
        future_time_feat,
        mode
    ):
        if mode == 'all':
            time_feat = torch.cat(
                (past_time_feat[:, -self.max_context_length:, ...], future_time_feat), dim=1)
        elif mode == 'encode':
            time_feat = past_time_feat[:, -self.max_context_length:, ...]
        elif mode == 'decode':
            time_feat = future_time_feat
        return time_feat

    def get_inputs(self, batch_data, mode):
        inputs_list = []

        input_seq = self.get_input_sequence(
            batch_data.past_target_cdf, batch_data.future_target_cdf, mode=mode)
        input_length = input_seq.shape[1] # [B L n_lags*K]
        inputs_list.append(input_seq)

        if self.use_feat_idx_emb:
            input_feat_idx_emb = self.get_input_feat_idx_emb(
                batch_data.target_dimension_indicator, input_length) # [B L K*D]
            inputs_list.append(input_feat_idx_emb)

        if self.use_time_feat:
            input_time_feat = self.get_input_time_feat(
                batch_data.past_time_feat, batch_data.future_time_feat, mode=mode) # [B L Dt]
            inputs_list.append(input_time_feat)
        return torch.cat(inputs_list, dim=-1).to(dtype=torch.float32)
    
    def get_scale(self, batch_data):
        self.scaler.fit(
            batch_data.past_target_cdf[:, -self.max_context_length:, ...],
            batch_data.past_observed_values[:, -self.max_context_length:, ...]
        )
    
    def get_weighted_loss(self, batch_data, loss):
        observed_values =  batch_data.future_observed_values
        loss_weights, _ = observed_values.min(dim=-1, keepdim=True)
        loss = weighted_average(loss, weights=loss_weights, dim=1)
        return loss
    
    def loss(self, batch_data):
        raise NotImplementedError
    
    def forecast(self, batch_data=None, num_samples=None):
        raise NotImplementedError


class ConvertedParams:
    def __init__(self, params):
        for key, value in params.items():
            setattr(self, key, value)


class k2VAEModel(Forecaster):
    def __init__(
            self,
            d_model,
            d_ff,
            e_layers,
            dropout,
            activation,
            n_heads,
            factor,
            patch_len,
            multistep,
            dynamic_dim,
            hidden_layers,
            hidden_dim,
            weight_beta=0.001,
            sample_schedule=5,
            init_kalman='identity',
            init_koopman='both',
            **kwargs
    ):
        """
        Initialize the model with parameters.
        """
        super().__init__(**kwargs)
        # Initialize model parameters here
        config = ConvertedParams(kwargs)
        config.d_model = d_model
        config.d_ff = d_ff
        config.hidden_layers = hidden_layers
        config.dropout = dropout
        config.activation = activation
        config.e_layers = e_layers
        config.n_heads = n_heads
        config.factor = factor
        config.patch_len = patch_len
        config.multistep = multistep
        config.dynamic_dim = dynamic_dim
        config.hidden_dim = hidden_dim
        config.n_vars = self.input_size
        config.seq_len = self.context_length
        config.pred_len = self.prediction_length
        self.weight_beta = weight_beta

        config.sample_schedule = sample_schedule
        config.init_kalman = init_kalman
        config.init_koopman = init_koopman

        self.model = k2VAE(config)

    def forward(self, input):
        """
        Forward pass for the model.

        Parameters:
        inputs [Tensor]: Input tensor for the model.

        Returns:
        Tensor: Output tensor.
        """
        # Perform the forward pass of the model
        rec, prior_dist, post_dist = self.model(input)
        return rec, prior_dist, post_dist

    def kld_loss(self, dist):
        # Extract the mean and covariance matrix from the distribution
        mu_q = dist.loc  # Shape: (B, P, hidden)
        cov_q = dist.covariance_matrix  # Shape: (B, P, hidden, hidden)
        # Compute the log determinant of the covariance matrix
        log_det_q = torch.linalg.slogdet(cov_q)[1]  # Shape: (B, P)

        # Compute the trace of the covariance matrix (sum of diagonal elements)
        trace_q = torch.einsum('btii->bt', cov_q)  # Shape: (B, P)

        # Compute the squared norm of the mean for each time step
        mu_term = torch.sum(mu_q ** 2, dim=-1)  # Shape: (B, P)

        # Get the latent space dimension (256 in this case)
        latent_dim = mu_q.size(-1)

        # Compute the KL divergence for each time step
        kld = 0.5 * (trace_q + mu_term - latent_dim - log_det_q)  # Shape: (B, P)

        # Take the mean over the time steps, resulting in one KL value per batch sample
        kld = kld.mean(dim=-1)  # Shape: (B,)

        # Take the mean over the batch to get the final average KL divergence
        kld = kld.mean()  # Scalar

        return kld

    def loss(self, batch_data, threshold=1e2):
        """
        Compute the loss for the given batch data.

        Parameters:
        batch_data [dict]: Dictionary containing input data and possibly target data.

        Returns:
        Tensor: Computed loss.
        """
        # Extract inputs and targets from batch_data
        self.model.train()
        input = batch_data.past_target_cdf[:, -self.context_length:, :]
        target = batch_data.future_target_cdf
        # Forward pass
        rec, prior_dist, post_dist = self.forward(input)
        rec_loss = F.mse_loss(rec, input) + F.mse_loss(post_dist.loc, target)
        post_loss = -post_dist.log_prob(target).mean()
        kld_loss = self.kld_loss(prior_dist)
        weight_alpha = 1 if post_loss < threshold else 0
        # stabilize training process
        loss = rec_loss + weight_alpha * post_loss + self.weight_beta * kld_loss
        return loss

    def sample_from_distribution(self, input, num_samples):
        samples = self.model.sample(input, num_samples)
        return rearrange(samples, 'n b l c -> b n l c')

    def forecast(self, batch_data, num_samples=None):
        """
        Generate forecasts for the given batch data.

        Parameters:
        batch_data [dict]: Dictionary containing input data.
        num_samples [int, optional]: Number of samples per distribution during evaluation. Defaults to None.

        Returns:
        Tensor: Forecasted outputs.
        """
        # Perform the forward pass to get the outputs
        self.model.eval()
        input = batch_data.past_target_cdf[:, -self.context_length:, :]
        with torch.no_grad():
            if num_samples is not None:
                # If num_samples is specified, use it to sample from the distribution
                outputs = self.sample_from_distribution(input, num_samples)
            else:
                outputs = None
        return outputs  # [batch_size, num_samples, prediction_length, var_num]

In [ ]:
class TimeSeriesBatch:
    def __init__(self, past_target_cdf, future_target_cdf):
        self.past_target_cdf = past_target_cdf
        self.future_target_cdf = future_target_cdf

class SlidingWindowDataset(Dataset):
    def __init__(self, data, context_length, prediction_length):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.total_len = len(self.data) - context_length - prediction_length + 1

    def __len__(self):
        return max(0, self.total_len)

    def __getitem__(self, idx):
        past = self.data[idx : idx + self.context_length]
        future = self.data[idx + self.context_length : idx + self.context_length + self.prediction_length]
        return past, future

def collate_fn(batch):
    past_list = [item[0] for item in batch]
    future_list = [item[1] for item in batch]
    past_tensor = torch.stack(past_list, dim=0) # [B, context_len, n_vars]
    future_tensor = torch.stack(future_list, dim=0) # [B, pred_len, n_vars]
    return TimeSeriesBatch(past_tensor, future_tensor)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv(data_path)
df = df.fillna(0.0)

NUM_NODES = df.shape[1]
P_LAG = 60
T_PRED = 30
BATCH_SIZE = 64
q_80 = 0.8
alpha = 0.05

train_values = df.values[:-T_PRED]
val_values = df.values[-T_PRED:]

# Setup datasets and dataloaders
train_dataset = SlidingWindowDataset(train_values, context_length=P_LAG, prediction_length=T_PRED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# Instantiate k2VAEModel
model = k2VAEModel(
    target_dim=11,
    freq="D",
    d_model=64,
    d_ff=128,
    e_layers=2,
    dropout=0.1,
    activation='gelu',
    n_heads=4,
    factor=1,
    patch_len=10,        
    multistep=T_PRED,
    dynamic_dim=64,
    hidden_layers=2,
    hidden_dim=128,
    weight_beta=0.001,
    sample_schedule=5,
    init_kalman='identity',
    init_koopman='both',
    input_size=NUM_NODES,
    context_length=P_LAG, 
    prediction_length=T_PRED
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 100

model.train()
start_time = time.time()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        # Move batch data to device
        batch.past_target_cdf = batch.past_target_cdf.to(device)
        batch.future_target_cdf = batch.future_target_cdf.to(device)
        
        optimizer.zero_grad()
        loss = model.loss(batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss / max(1, len(train_loader)):.4f}")
training_time = time.time() - start_time

# Metrics
model.eval()
start_time = time.time()
val_tensor = torch.tensor(val_values, dtype=torch.float32).to(device)
context_tensor = torch.tensor(train_values[-P_LAG:], dtype=torch.float32).unsqueeze(0).to(device)

eval_batch = TimeSeriesBatch(
    past_target_cdf=context_tensor,
    future_target_cdf=val_tensor.unsqueeze(0)
)

NUM_SAMPLES = 100
with torch.no_grad():
    # Shape: [batch_size=1, num_samples, prediction_length, var_num]
    forecast_samples = model.forecast(eval_batch, num_samples=NUM_SAMPLES)

inference_time = time.time()-start_time

y_preds_ensemble_t = forecast_samples.squeeze(0).cpu() # [num_samples, prediction_length, var_num]
y_true_t = val_tensor.cpu()                             # [prediction_length, var_num]
y_train_t = torch.tensor(train_values, dtype=torch.float32) # [train_len, var_num]

y_pred_mean_t = y_preds_ensemble_t.mean(dim=0)
y_pred_80_t = torch.quantile(y_preds_ensemble_t, q_80, dim=0)
y_lower_t = torch.quantile(y_preds_ensemble_t, alpha / 2, dim=0)
y_upper_t = torch.quantile(y_preds_ensemble_t, 1 - (alpha / 2), dim=0)

emp_cov = empirical_coverage(y_true_t.numpy(), y_lower_t.numpy(), y_upper_t.numpy())
wink_score = winkler_score(y_true_t.numpy(), y_lower_t.numpy(), y_upper_t.numpy(), alpha)

print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80):.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 12. Transformer-MAF

In [ ]:
# Some patching and workarounds required because pytorch-ts library is no longer updated

file_path = '../pytorch-ts/pts/__init__.py'
with open(file_path, 'r') as f:
    lines = f.readlines()

# Write it back without the broken pkg_resources import
with open(file_path, 'w') as f:
    for line in lines:
        if 'pkg_resources' in line:
            continue
        if 'get_distribution' in line:
            line = line.replace('get_distribution(__name__).version', '"0.6.0"')
        f.write(line)

file_path = '../pytorch-ts/pts/model/transformer_tempflow/transformer_tempflow_estimator.py'

with open(file_path, 'r') as f:
    content = f.read()

# Remove the invalid 'freq=self.freq,' argument
patched_content = content.replace('freq=self.freq,', '')

# Write the fixed code back
with open(file_path, 'w') as f:
    f.write(patched_content)

print("Patches successful!")

In [ ]:
from pts.model.transformer_tempflow import TransformerTempFlowEstimator
from pts import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv(data_path)
df = df.fillna(0.0)

NUM_NODES = df.shape[1]
P_LAG = 60
T_PRED = 30
BATCH_SIZE = 64
q_80 = 0.8
alpha = 0.05

train_values = df.values[:-T_PRED]
val_values = df.values[-T_PRED:]
train_target = train_values.T.astype(np.float32)
train_target += np.random.normal(0, 1e-6, train_target.shape).astype(np.float32)

test_target = df.values.T.astype(np.float32)

freq = "D"  
start_date = pd.Period("2020-01-01", freq=freq)


train_ds = ListDataset(
    [{FieldName.TARGET: train_target, FieldName.START: start_date}] * 500,
    freq=freq,
    one_dim_target=False
)

# test_ds does not need to be multiplied, as the evaluator extracts deterministically
test_ds = ListDataset(
    [{FieldName.TARGET: test_target, FieldName.START: start_date}],
    freq=freq,
    one_dim_target=False
)


EXPECTED_INPUT_SIZE = 46 # Dataset-specific, needs to be adjusted. This is specific to Belgium data

estimator = TransformerTempFlowEstimator(
    d_model=120,
    num_heads=4,
    input_size=EXPECTED_INPUT_SIZE,
    target_dim=NUM_NODES,
    prediction_length=T_PRED,
    context_length=P_LAG,
    flow_type='MAF',
    dequantize=False, 
    freq=freq,
    scaling=True,    
    trainer=Trainer(
        device=device,
        epochs=100,
        learning_rate=1e-3,
        num_batches_per_epoch=50,
        batch_size=BATCH_SIZE,
    )
)

print("Training Transformer-MAF...")
start_train = time.time()

predictor = estimator.train(
    train_ds, 
    num_workers=0, 
    prefetch_factor=None
)

training_time = time.time() - start_train

print("Generating probabilistic forecasts...")
start_inf = time.time()

forecast_it, _ = make_evaluation_predictions(
    dataset=test_ds,
    predictor=predictor,
    num_samples=100
)
forecasts = list(forecast_it)
inference_time = time.time() - start_inf

y_preds_ensemble_np = forecasts[0].samples
y_preds_ensemble_t = torch.tensor(y_preds_ensemble_np, dtype=torch.float32).to(device)

y_pred_median_t = torch.median(y_preds_ensemble_t, dim=0).values
lower_bounds_t = torch.quantile(y_preds_ensemble_t, q=alpha / 2, dim=0)
upper_bounds_t = torch.quantile(y_preds_ensemble_t, q=1 - alpha / 2, dim=0)
y_pred_80_t = torch.quantile(y_preds_ensemble_t, q=q_80, dim=0)

lower_bounds_np = lower_bounds_t.cpu().numpy()
upper_bounds_np = upper_bounds_t.cpu().numpy()
y_pred_median_np = y_pred_median_t.cpu().numpy()

emp_cov = empirical_coverage(val_values, lower_bounds_np, upper_bounds_np)
wink_score = winkler_score(val_values, lower_bounds_np, upper_bounds_np, alpha)

y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)

print("\n--- Metrics ---")
print(f"{mae(y_true_t, y_pred_median_t):.4f}")
print(f"{mase(y_true_t, y_pred_median_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_median_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80):.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 13. TimeGrad

In [ ]:
from pts.model.time_grad import TimeGradEstimator
from pts import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv(data_path)
df = df.fillna(0.0)

NUM_NODES = df.shape[1]
P_LAG = 60
T_PRED = 30
BATCH_SIZE = 64
q_80 = 0.8
alpha = 0.05

train_values = df.values[:-T_PRED]
val_values = df.values[-T_PRED:]

train_target = train_values.T.astype(np.float32)
train_target += np.random.normal(0, 1e-6, train_target.shape).astype(np.float32)

test_target = df.values.T.astype(np.float32)

freq = "D"  
start_date = pd.Period("2020-01-01", freq=freq)

train_ds = ListDataset(
    [{FieldName.TARGET: train_target, FieldName.START: start_date}] * 500,
    freq=freq,
    one_dim_target=False
)

test_ds = ListDataset(
    [{FieldName.TARGET: test_target, FieldName.START: start_date}],
    freq=freq,
    one_dim_target=False
)

NUM_NODES = df.shape[1]

estimator = TimeGradEstimator(
    target_dim=NUM_NODES,
    prediction_length=T_PRED,
    context_length=P_LAG,
    cell_type='GRU',
    input_size=46,          # Dataset-specific
    freq=freq,
    loss_type='l2',
    scaling=True,
    #diff_steps=100,
    #beta_end=0.1,
    beta_schedule="linear",
    trainer=Trainer(
        device=device,
        epochs=100,
        learning_rate=1e-3,
        #grad_clip_norm=1.0,
        num_batches_per_epoch=30,
        batch_size=BATCH_SIZE,
    )
)

print("Training TimeGrad on Belgium Data...")
start_train = time.time()
predictor = estimator.train(train_ds, num_workers=0, prefetch_factor=None)
training_time = time.time() - start_train

print("Generating probabilistic forecasts...")
start_inf = time.time()

forecast_it, _ = make_evaluation_predictions(
    dataset=test_ds,
    predictor=predictor,
    num_samples=100
)
forecasts = list(forecast_it)
inference_time = time.time() - start_inf

y_preds_ensemble_np = forecasts[0].samples
y_preds_ensemble_t = torch.tensor(y_preds_ensemble_np, dtype=torch.float32).to(device)

y_pred_median_t = torch.median(y_preds_ensemble_t, dim=0).values

lower_bounds_t = torch.quantile(y_preds_ensemble_t, q=alpha / 2, dim=0)
upper_bounds_t = torch.quantile(y_preds_ensemble_t, q=1 - alpha / 2, dim=0)
y_pred_80_t = torch.quantile(y_preds_ensemble_t, q=q_80, dim=0)

lower_bounds_np = lower_bounds_t.cpu().numpy()
upper_bounds_np = upper_bounds_t.cpu().numpy()
y_pred_median_np = y_pred_median_t.cpu().numpy()

emp_cov = empirical_coverage(val_values, lower_bounds_np, upper_bounds_np)
wink_score = winkler_score(val_values, lower_bounds_np, upper_bounds_np, alpha)

y_true_t = torch.tensor(val_values, dtype=torch.float32).to(device)
y_train_t = torch.tensor(train_values, dtype=torch.float32).to(device)

print("\n--- Metrics ---")
print(f"{mae(y_true_t, y_pred_median_t):.4f}")
print(f"{mase(y_true_t, y_pred_median_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_median_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80):.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")


In [ ]:
# Note: If the above cell throws an error, run this cell, restart session, then re-run
!sed -i '/freq=self.freq/d' ../pytorch-ts/pts/model/time_grad/time_grad_estimator.py

# 14. TimesFM-2.5

In [ ]:
df = pd.read_csv(data_path)

horizon = 30
num_nodes = df.shape[1]
q_80 = 0.8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract arrays for evaluation (shape: time, num_nodes)
train_values = df.iloc[:-horizon].values
val_values = df.iloc[-horizon:].values

# Convert to Darts TimeSeries
train_ts = TimeSeries.from_values(train_values)

# Quantiles should be subset of the ones with which TimesFM was trained in darts
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
model = TimesFM2p5Model(input_chunk_length=60, output_chunk_length=30,
                     likelihood=QuantileRegression(quantiles=quantiles))

t0 = time.time()
model.fit(train_ts)
training_time = time.time() - t0
num_samples = 100
t0 = time.time()
pred_ts = model.predict(n=horizon, series=train_ts, num_samples=num_samples)
inference_time = time.time() - t0
pred_samples = pred_ts.all_values() 

# Rearrange to (samples, time, components) for easier quantile calculations
y_preds_ensemble_np = np.transpose(pred_samples, (2, 0, 1))

# Compute point predictions (mean) and quantiles
y_pred_np = np.mean(y_preds_ensemble_np, axis=0) # Mean as point forecast
lower_bounds_np = np.quantile(y_preds_ensemble_np, 0.025, axis=0) 
upper_bounds_np = np.quantile(y_preds_ensemble_np, 0.975, axis=0) 
y_pred_80_np = np.quantile(y_preds_ensemble_np, q_80, axis=0)   # 80th percentile

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32, device=device)
y_train_t = torch.tensor(train_values, dtype=torch.float32, device=device)
y_pred_mean_t = torch.tensor(y_pred_np, dtype=torch.float32, device=device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble_np, dtype=torch.float32, device=device)
y_pred_80_t = torch.tensor(y_pred_80_np, dtype=torch.float32, device=device)
alpha = 0.05 
emp_cov = empirical_coverage(val_values, lower_bounds_np, upper_bounds_np)
wink_score = winkler_score(val_values, lower_bounds_np, upper_bounds_np, alpha)

print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 15. Chronos-2

In [ ]:
df = pd.read_csv(data_path)

horizon = 30
num_nodes = df.shape[1]
q_80 = 0.8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract arrays for evaluation (shape: time, num_nodes)
train_values = df.iloc[:-horizon].values
val_values = df.iloc[-horizon:].values

# Convert to Darts TimeSeries
train_ts = TimeSeries.from_values(train_values)

# Quantiles should be subset of the ones with which TimesFM was trained in darts
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
model = Chronos2Model(input_chunk_length=60, output_chunk_length=30,
                     likelihood=QuantileRegression(quantiles=quantiles))

t0 = time.time()
model.fit(train_ts)
training_time = time.time() - t0
num_samples = 100
t0 = time.time()
pred_ts = model.predict(n=horizon, series=train_ts, num_samples=num_samples)
inference_time = time.time() - t0
pred_samples = pred_ts.all_values() 

# Rearrange to (samples, time, components) for easier quantile calculations
y_preds_ensemble_np = np.transpose(pred_samples, (2, 0, 1))

# Compute point predictions (mean) and quantiles
y_pred_np = np.mean(y_preds_ensemble_np, axis=0) # Mean as point forecast
lower_bounds_np = np.quantile(y_preds_ensemble_np, 0.025, axis=0) 
upper_bounds_np = np.quantile(y_preds_ensemble_np, 0.975, axis=0) 
y_pred_80_np = np.quantile(y_preds_ensemble_np, q_80, axis=0)   # 80th percentile

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32, device=device)
y_train_t = torch.tensor(train_values, dtype=torch.float32, device=device)
y_pred_mean_t = torch.tensor(y_pred_np, dtype=torch.float32, device=device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble_np, dtype=torch.float32, device=device)
y_pred_80_t = torch.tensor(y_pred_80_np, dtype=torch.float32, device=device)
alpha = 0.05 
emp_cov = empirical_coverage(val_values, lower_bounds_np, upper_bounds_np)
wink_score = winkler_score(val_values, lower_bounds_np, upper_bounds_np, alpha)

print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")

# 16. PatchTSTFM

In [ ]:
df = pd.read_csv(data_path)

horizon = 30
num_nodes = df.shape[1]
q_80 = 0.8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract arrays for evaluation (shape: time, num_nodes)
train_values = df.iloc[:-horizon].values
val_values = df.iloc[-horizon:].values

# Convert to Darts TimeSeries
train_ts = TimeSeries.from_values(train_values)

# Quantiles should be subset of the ones with which TimesFM was trained in darts
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
model = PatchTSTFMModel(input_chunk_length=60, output_chunk_length=30,
                     likelihood=QuantileRegression(quantiles=quantiles))

t0 = time.time()
model.fit(train_ts)
training_time = time.time() - t0
num_samples = 100
t0 = time.time()
pred_ts = model.predict(n=horizon, series=train_ts, num_samples=num_samples)
inference_time = time.time() - t0
pred_samples = pred_ts.all_values() 

# Rearrange to (samples, time, components) for easier quantile calculations
y_preds_ensemble_np = np.transpose(pred_samples, (2, 0, 1))

# Compute point predictions (mean) and quantiles
y_pred_np = np.mean(y_preds_ensemble_np, axis=0) # Mean as point forecast
lower_bounds_np = np.quantile(y_preds_ensemble_np, 0.025, axis=0) 
upper_bounds_np = np.quantile(y_preds_ensemble_np, 0.975, axis=0) 
y_pred_80_np = np.quantile(y_preds_ensemble_np, q_80, axis=0)   # 80th percentile

# Metrics
y_true_t = torch.tensor(val_values, dtype=torch.float32, device=device)
y_train_t = torch.tensor(train_values, dtype=torch.float32, device=device)
y_pred_mean_t = torch.tensor(y_pred_np, dtype=torch.float32, device=device)
y_preds_ensemble_t = torch.tensor(y_preds_ensemble_np, dtype=torch.float32, device=device)
y_pred_80_t = torch.tensor(y_pred_80_np, dtype=torch.float32, device=device)
alpha = 0.05 
emp_cov = empirical_coverage(val_values, lower_bounds_np, upper_bounds_np)
wink_score = winkler_score(val_values, lower_bounds_np, upper_bounds_np, alpha)

print(f"{mae(y_true_t, y_pred_mean_t):.4f}")
print(f"{mase(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{rmsse(y_true_t, y_pred_mean_t, y_train_t):.4f}")
print(f"{emp_cov:.4f}")
print(f"{crps_approximation(y_true_t, y_preds_ensemble_t):.4f}")
print(f"{wink_score:.4f}")
print(f"{pinball_loss(y_true_t, y_pred_80_t, quantile=q_80).item():.4f}")
print(f"{training_time:.4f}")
print(f"{inference_time:.4f}")